In [ ]:
import os
import json
import pandas as pd
import numpy as np

base_score_dict = {
    "pickscore": 0.7930436655282974*26,
    "imagereward": 0.16912677358626388,
    "hpsv3": 5.892178298220038,
    "deqa": 3.745064453125,
    "ma-agiqa": 2.293051694869995,
    "aesthetic": 4.662666016578674
}

inpainting_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-v1-5/generate_images/pick_a_pic_v2/random_9984_images_no_anime_pickscore_002-hpdv3_all-inpainting"
text2image_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-v1-5/generate_images/pick_a_pic_v2/random_9984_images_no_anime_pickscore_002-hpdv3_all-t2i"
ckpt_list = [ 100, 200, 300, 400, 500, 600, 700, 800 ]



In [ ]:
# 读取每个 checkpoint 的得分数据
inpainting_scores = {}
text2image_scores = {}

for ckpt in ckpt_list:
    # 读取 inpainting 得分
    inpainting_path = os.path.join(inpainting_score_dir, f"ckpt-{ckpt}", "average_scores.json")
    if os.path.exists(inpainting_path):
        with open(inpainting_path, 'r') as f:
            inpainting_scores[ckpt] = json.load(f)
            # pickscore 值需要乘以 26
            if 'pickscore' in inpainting_scores[ckpt]:
                inpainting_scores[ckpt]['pickscore'] = inpainting_scores[ckpt]['pickscore'] * 26
    else:
        print(f"Warning: {inpainting_path} not found")
        inpainting_scores[ckpt] = {}
    
    # 读取 text2image 得分
    text2image_path = os.path.join(text2image_score_dir, f"ckpt-{ckpt}", "average_scores.json")
    if os.path.exists(text2image_path):
        with open(text2image_path, 'r') as f:
            text2image_scores[ckpt] = json.load(f)
            # pickscore 值需要乘以 26
            if 'pickscore' in text2image_scores[ckpt]:
                text2image_scores[ckpt]['pickscore'] = text2image_scores[ckpt]['pickscore'] * 26
    else:
        print(f"Warning: {text2image_path} not found")
        text2image_scores[ckpt] = {}

print("数据读取完成！")
print(f"Inpainting scores keys: {list(inpainting_scores.keys())}")
print(f"Text2Image scores keys: {list(text2image_scores.keys())}")


In [ ]:
all_metrics = ['pickscore', 'imagereward', 'hpsv3', 'deqa', 'ma-agiqa', 'aesthetic']
print(f"Used metrics: {all_metrics}")

comparison_data = []

# 首先添加 checkpoint=0 的 base_score 数据
for metric in all_metrics:
    base_val = base_score_dict.get(metric, np.nan)
    comparison_data.append({
        'checkpoint': 0,
        'metric': metric,
        'inpainting': base_val,
        'text2image': base_val,
        'difference': 0.0,
        'relative_diff': 0.0
    })

for ckpt in ckpt_list:
    for metric in all_metrics:
        inpainting_val = inpainting_scores.get(ckpt, {}).get(metric, np.nan)
        text2image_val = text2image_scores.get(ckpt, {}).get(metric, np.nan)
        diff = inpainting_val - text2image_val if not (np.isnan(inpainting_val) or np.isnan(text2image_val)) else np.nan
        
        comparison_data.append({
            'checkpoint': ckpt,
            'metric': metric,
            'inpainting': inpainting_val,
            'text2image': text2image_val,
            'difference': diff,
            'relative_diff': (diff / text2image_val * 100) if not (np.isnan(diff) or np.isnan(text2image_val) or text2image_val == 0) else np.nan
        })

df_comparison = pd.DataFrame(comparison_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.ticker import MaxNLocator, FormatStrFormatter, LinearLocator

# ========== 绘图参数配置 ==========
# 线条和标记样式参数
LINE_WIDTH = 4.0              # 线条宽度
MARKER_SIZE = 12              # 标记大小
MARKER_EDGE_WIDTH = 2.0       # 标记边框宽度
BASELINE_WIDTH = 2.5          # 基准线宽度

# 字体大小参数
TITLE_FONTSIZE = 14           # 子图标题字体大小
AXIS_LABEL_FONTSIZE = 12      # 坐标轴标签字体大小
LEGEND_FONTSIZE = 16          # 图例字体大小
LEGEND_MARKER_SIZE = 18       # 图例标记大小

# ===================================

# 设置 seaborn 样式
sns.set_style("whitegrid")

colors = {
    'Inpainting': '#fa7f72',
    'Text2Image': '#7b9ed7'
}

# 指标名称显示映射
metric_display_names = {
    'pickscore': 'PickScore',
    'imagereward': 'ImageReward',
    'hpsv3': 'HPSv3',
    'deqa': 'DeQA',
    'ma-agiqa': 'MA-AGIQA',
    'aesthetic': 'Aesthetic'
}

# 数据准备：将宽格式转换为长格式（seaborn 更适合长格式）
df_long = df_comparison.melt(
    id_vars=['checkpoint', 'metric'],
    value_vars=['inpainting', 'text2image'],
    var_name='model',
    value_name='score'
)

# 美化模型名称
df_long['model'] = df_long['model'].map({
    'inpainting': 'Inpainting',
    'text2image': 'Text2Image'
})

# 创建子图 - 只绘制三个指标，改为1行3列
fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
axes = axes.flatten()

# 只绘制这三个指标
all_metrics = ['imagereward', 'hpsv3', 'deqa']

for idx, metric in enumerate(all_metrics):
    # 筛选当前指标的数据
    metric_data = df_long[df_long['metric'] == metric]
    
    # 使用 seaborn 绘制线图
    sns.lineplot(
        data=metric_data,
        x='checkpoint',
        y='score',
        hue='model',
        style='model',
        markers=['o', 's'],
        markersize=MARKER_SIZE,
        linewidth=LINE_WIDTH,
        palette=colors,
        ax=axes[idx],
        legend=False,
        markeredgecolor='white',
        markeredgewidth=MARKER_EDGE_WIDTH
    )
    
    # 设置标题和标签
    metric_display = metric_display_names.get(metric, metric.upper())
    axes[idx].set_title(f'({chr(97+idx)}) {metric_display}', 
                       fontsize=TITLE_FONTSIZE, fontweight='semibold', pad=12)
    axes[idx].set_xlabel('Step', fontsize=AXIS_LABEL_FONTSIZE, fontweight='medium')
    axes[idx].set_ylabel('Score', fontsize=AXIS_LABEL_FONTSIZE, fontweight='medium')
    
    # 设置 x 轴和 y 轴都只显示 5 个刻度
    axes[idx].xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    # 使用 LinearLocator 确保 y 轴精确显示 5 个刻度
    axes[idx].yaxis.set_major_locator(LinearLocator(numticks=5))
    # 设置 y 轴刻度格式为两位小数
    axes[idx].yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    # 美化网格
    axes[idx].grid(True, alpha=0.25, linestyle='--', linewidth=0.8)
    axes[idx].set_axisbelow(True)
    
    # 移除上边框和右边框
    sns.despine(ax=axes[idx], top=True, right=True)

# 创建统一图例
handles = [
    plt.Line2D([0], [0], marker='o', color=colors['Inpainting'], 
               label='Saliency Inpainting', linewidth=LINE_WIDTH, markersize=LEGEND_MARKER_SIZE,
               markerfacecolor=colors['Inpainting'], 
               markeredgecolor='white', markeredgewidth=MARKER_EDGE_WIDTH),
    plt.Line2D([0], [0], marker='s', color=colors['Text2Image'], 
               label='Text2Image', linewidth=LINE_WIDTH, markersize=LEGEND_MARKER_SIZE,
               markerfacecolor=colors['Text2Image'], 
               markeredgecolor='white', markeredgewidth=MARKER_EDGE_WIDTH)
]

fig.legend(handles=handles, loc='center', ncol=2, 
          frameon=True, fontsize=LEGEND_FONTSIZE, bbox_to_anchor=(0.5, 0.99),
          framealpha=0.95, edgecolor='#cccccc', fancybox=True)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
